# Bit Accuracy Analysis & Gamma–Image Quality Correlation

This notebook explores two questions:

1. **How does bit accuracy relate to each benchmark parameter?** (gamma, message length, transform type, transform intensity)
2. **How does gamma affect generated image quality?** (PSNR/SSIM between original and stego image — *before* any attack)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
from pathlib import Path
from scipy import stats as sp_stats

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams["figure.dpi"] = 130
SAVE_DIR = Path("../results/sdxl/")

CSV_PATH = SAVE_DIR / "benchmark_results.csv"
assert CSV_PATH.exists(), f"Results not found at {CSV_PATH}. Run benchmark_ldstega.py first."

df = pd.read_csv(CSV_PATH)
df["transform_params"] = df["transform_params"].apply(json.loads)

# Derived columns
df["bit_accuracy_pct"] = df["bit_accuracy"] * 100
# Ensure message_chars exists (bits / 8 = UTF-8 characters)
if "message_chars" not in df.columns:
    df["message_chars"] = df["message_length"] // 8

# Helper: format "N bits (M chars)" for labels
def bits_chars_label(bits):
    return f"{bits} bits ({bits // 8} chars)"

# Convenience subsets
ind = df[df["suite"] == "individual"].copy()
msg = df[df["suite"] == "messenger"].copy()
ps  = df[df["suite"] == "param_sweep"].copy()
strs = df[df["suite"] == "stress"].copy()

print(f"Loaded {len(df)} rows  |  suites: {df['suite'].unique().tolist()}")
print(f"Gamma values present: {sorted(df['gamma'].unique())}")
msg_lens = sorted(df["message_length"].unique())
print(f"Message lengths: {[bits_chars_label(b) for b in msg_lens]}")
df.head(3)

---
# Part 1 — Bit Accuracy vs Parameters

## 1.1  Bit Accuracy Distribution by Suite

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
sns.violinplot(data=df, x="suite", y="bit_accuracy_pct", inner="quartile",
               palette="Set2", ax=axes[0], cut=0)
axes[0].set_ylabel("Bit Accuracy (%)")
axes[0].set_xlabel("")
axes[0].set_title("Bit Accuracy Distribution by Suite")
axes[0].axhline(y=50, color="red", ls="--", alpha=0.4, label="Random guess (50%)")
axes[0].legend(fontsize=9)

# Box plot (shows outliers better)
sns.boxplot(data=df, x="suite", y="bit_accuracy_pct",
            palette="Set2", ax=axes[1], fliersize=3)
axes[1].set_ylabel("Bit Accuracy (%)")
axes[1].set_xlabel("")
axes[1].set_title("Bit Accuracy — Box Plot")
axes[1].axhline(y=50, color="red", ls="--", alpha=0.4)

plt.tight_layout()
plt.savefig(SAVE_DIR / "bit_acc_distribution_by_suite.png", bbox_inches="tight")
plt.show()

## 1.2  Bit Accuracy vs Gamma (param sweep, fixed JPEG q=75)

In [ ]:
if len(ps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: one line per message length ---
    for ml in sorted(ps["message_length"].unique()):
        sub = ps[ps["message_length"] == ml]
        agg = sub.groupby("gamma")["bit_accuracy_pct"].agg(["mean", "std"]).reset_index()
        axes[0].errorbar(agg["gamma"], agg["mean"], yerr=agg["std"],
                         marker="o", capsize=3, label=bits_chars_label(ml))

    axes[0].set_xlabel("Gamma (\u03b3)")
    axes[0].set_ylabel("Bit Accuracy (%)")
    axes[0].set_title("Bit Accuracy vs \u03b3  (JPEG q=75)")
    axes[0].legend(title="message size", fontsize=7, title_fontsize=8)
    axes[0].axhline(y=50, color="red", ls="--", alpha=0.3)

    # --- Right: one line per gamma ---
    for g in sorted(ps["gamma"].unique()):
        sub = ps[ps["gamma"] == g]
        agg = sub.groupby("message_length")["bit_accuracy_pct"].agg(["mean", "std"]).reset_index()
        axes[1].errorbar(agg["message_length"], agg["mean"], yerr=agg["std"],
                         marker="s", capsize=3, label=f"\u03b3={g}")

    axes[1].set_xlabel("Message Length (bits)\n" +
                        "  |  ".join([f"{b}b={b//8}ch" for b in sorted(ps["message_length"].unique())]))
    axes[1].set_ylabel("Bit Accuracy (%)")
    axes[1].set_title("Bit Accuracy vs Message Length  (JPEG q=75)")
    axes[1].set_xscale("log", base=2)
    axes[1].xaxis.set_major_formatter(mticker.ScalarFormatter())
    axes[1].legend(fontsize=8)
    axes[1].axhline(y=50, color="red", ls="--", alpha=0.3)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / "bit_acc_vs_gamma_and_msglen.png", bbox_inches="tight")
    plt.show()
else:
    print("No param_sweep data.")

## 1.3  Bit Accuracy vs Transform Intensity (per family)

In [ ]:
if len(ind) > 0:
    # Build (family, intensity, bit_accuracy) from individual suite
    records = []
    for _, row in ind.iterrows():
        p = row["transform_params"]
        t = p.get("transform", "none")
        # Choose the main intensity parameter per family
        if t == "jpeg":       intensity = p["quality"]
        elif t == "webp":     intensity = p["quality"]
        elif t == "resize":   intensity = p["scale"]
        elif t == "gaussian_blur": intensity = p["sigma"]
        elif t == "gaussian_noise": intensity = p["std"]
        elif t == "brightness":  intensity = abs(p["brightness"])
        elif t == "contrast":    intensity = abs(p["contrast"])
        elif t == "saturation":  intensity = abs(p["saturation"])
        elif t == "center_crop": intensity = p["ratio"]
        elif t == "rotation":    intensity = p["angle"]
        else: continue
        records.append({"family": t, "intensity": intensity,
                        "bit_accuracy_pct": row["bit_accuracy"] * 100,
                        "seed": row["seed"]})

    idf = pd.DataFrame(records)

    families = sorted(idf["family"].unique())
    n = len(families)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes_flat = axes.flatten()

    for i, fam in enumerate(families):
        ax = axes_flat[i]
        sub = idf[idf["family"] == fam]
        agg = sub.groupby("intensity")["bit_accuracy_pct"].agg(["mean", "std"]).reset_index()
        ax.errorbar(agg["intensity"], agg["mean"], yerr=agg["std"],
                    marker="o", capsize=3, color=sns.color_palette("tab10")[i % 10])
        ax.set_title(fam, fontsize=11)
        ax.set_ylabel("Bit Accuracy (%)")
        ax.set_xlabel("intensity")
        ax.axhline(y=50, color="red", ls="--", alpha=0.3)
        ax.set_ylim(bottom=max(0, agg["mean"].min() - 15), top=105)

    # Hide unused axes
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle("Bit Accuracy vs Transform Intensity (Suite 1)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "bit_acc_vs_intensity_per_family.png", bbox_inches="tight")
    plt.show()
else:
    print("No individual suite data.")

## 1.4  Bit Accuracy vs Image Degradation (PSNR / SSIM of transformed image)

In [ ]:
# Use individual + messenger + stress suites (not param_sweep, which varies gamma)
attack_df = df[df["suite"].isin(["individual", "messenger", "stress"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PSNR_transformed vs bit_accuracy
sc0 = axes[0].scatter(
    attack_df["psnr_transformed"], attack_df["bit_accuracy_pct"],
    c=attack_df["suite"].map({"individual": 0, "messenger": 1, "stress": 2}),
    cmap="Set1", alpha=0.6, s=25, edgecolors="none",
)
# Regression line
mask = np.isfinite(attack_df["psnr_transformed"])
if mask.sum() > 2:
    slope, intercept, r, p, _ = sp_stats.linregress(
        attack_df.loc[mask, "psnr_transformed"], attack_df.loc[mask, "bit_accuracy_pct"])
    xs = np.linspace(attack_df["psnr_transformed"].min(), attack_df["psnr_transformed"].max(), 50)
    axes[0].plot(xs, intercept + slope * xs, "k--", alpha=0.5,
                 label=f"r={r:.3f}, p={p:.2e}")
    axes[0].legend(fontsize=9)
axes[0].set_xlabel("PSNR of transformed image (dB)")
axes[0].set_ylabel("Bit Accuracy (%)")
axes[0].set_title("Bit Accuracy vs Transform PSNR")
axes[0].axhline(y=50, color="red", ls="--", alpha=0.3)

# Right: SSIM_transformed vs bit_accuracy
sc1 = axes[1].scatter(
    attack_df["ssim_transformed"], attack_df["bit_accuracy_pct"],
    c=attack_df["suite"].map({"individual": 0, "messenger": 1, "stress": 2}),
    cmap="Set1", alpha=0.6, s=25, edgecolors="none",
)
mask2 = np.isfinite(attack_df["ssim_transformed"])
if mask2.sum() > 2:
    slope2, intercept2, r2, p2, _ = sp_stats.linregress(
        attack_df.loc[mask2, "ssim_transformed"], attack_df.loc[mask2, "bit_accuracy_pct"])
    xs2 = np.linspace(attack_df["ssim_transformed"].min(), attack_df["ssim_transformed"].max(), 50)
    axes[1].plot(xs2, intercept2 + slope2 * xs2, "k--", alpha=0.5,
                 label=f"r={r2:.3f}, p={p2:.2e}")
    axes[1].legend(fontsize=9)
axes[1].set_xlabel("SSIM of transformed image")
axes[1].set_ylabel("Bit Accuracy (%)")
axes[1].set_title("Bit Accuracy vs Transform SSIM")
axes[1].axhline(y=50, color="red", ls="--", alpha=0.3)

# Shared legend for suite colors
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=sns.color_palette("Set1")[i],
           markersize=7, label=s)
    for i, s in enumerate(["individual", "messenger", "stress"])
]
fig.legend(handles=legend_elems, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.04), fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR / "bit_acc_vs_psnr_ssim_transformed.png", bbox_inches="tight")
plt.show()

## 1.5  Bit Accuracy Heatmap — Gamma vs Message Length (param sweep)

In [ ]:
if len(ps) > 0:
    # Build pivot with dual-label columns: "bits (chars)"
    ps["msg_label"] = ps["message_length"].apply(bits_chars_label)
    label_order = [bits_chars_label(b) for b in sorted(ps["message_length"].unique())]

    pivot_acc = (
        ps.groupby(["gamma", "msg_label"])["bit_accuracy_pct"]
        .mean().reset_index()
        .pivot(index="gamma", columns="msg_label", values="bit_accuracy_pct")
    )[label_order]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(
        pivot_acc, annot=True, fmt=".1f", cmap="RdYlGn",
        vmin=50, vmax=100, linewidths=0.5, ax=ax,
        cbar_kws={"label": "Bit Accuracy (%)"},
    )
    ax.set_title("Bit Accuracy (%)  \u2014  \u03b3 vs Message Size  (JPEG q=75)")
    ax.set_ylabel("Truncation Gamma (\u03b3)")
    ax.set_xlabel("Message Size")
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "bit_acc_heatmap_gamma_msglen.png", bbox_inches="tight")
    plt.show()

    ps.drop(columns=["msg_label"], inplace=True)
else:
    print("No param_sweep data.")

## 1.6  Bit Accuracy Ranked by Messenger Platform

In [ ]:
if len(msg) > 0:
    fig, ax = plt.subplots(figsize=(9, 5))

    order = msg.groupby("name")["bit_accuracy_pct"].mean().sort_values(ascending=False).index.tolist()
    sns.barplot(data=msg, x="name", y="bit_accuracy_pct", order=order,
                palette="coolwarm_r", errorbar="sd", capsize=0.1, ax=ax)

    for i, name in enumerate(order):
        m = msg[msg["name"] == name]["bit_accuracy_pct"].mean()
        ax.text(i, m + 0.8, f"{m:.1f}%", ha="center", fontsize=9, fontweight="bold")

    ax.set_ylabel("Bit Accuracy (%)")
    ax.set_xlabel("")
    ax.set_title("Bit Accuracy per Messenger Platform")
    ax.axhline(y=50, color="red", ls="--", alpha=0.4, label="Random guess")
    ax.axhline(y=90, color="green", ls="--", alpha=0.4, label="90% threshold")
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "bit_acc_messenger_ranked.png", bbox_inches="tight")
    plt.show()
else:
    print("No messenger data.")

---
# Part 2 — Gamma vs Generated Image Quality

`psnr_stego` and `ssim_stego` measure the quality of the stego image compared to the original — **before any attack**. Higher gamma forces latent values further from the mean, which may degrade visual quality.

## 2.1  Gamma vs Stego Image Quality (PSNR & SSIM)

In [ ]:
if len(ps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- PSNR ---
    for ml in sorted(ps["message_length"].unique()):
        sub = ps[ps["message_length"] == ml]
        agg = sub.groupby("gamma")["psnr_stego"].agg(["mean", "std"]).reset_index()
        axes[0].errorbar(agg["gamma"], agg["mean"], yerr=agg["std"],
                         marker="o", capsize=3, label=bits_chars_label(ml))
    axes[0].set_xlabel("Gamma (\u03b3)")
    axes[0].set_ylabel("PSNR (dB) \u2014 stego vs original")
    axes[0].set_title("Stego Image Quality (PSNR) vs \u03b3")
    axes[0].legend(title="message size", fontsize=7, title_fontsize=8)

    # --- SSIM ---
    for ml in sorted(ps["message_length"].unique()):
        sub = ps[ps["message_length"] == ml]
        agg = sub.groupby("gamma")["ssim_stego"].agg(["mean", "std"]).reset_index()
        axes[1].errorbar(agg["gamma"], agg["mean"], yerr=agg["std"],
                         marker="s", capsize=3, label=bits_chars_label(ml))
    axes[1].set_xlabel("Gamma (\u03b3)")
    axes[1].set_ylabel("SSIM \u2014 stego vs original")
    axes[1].set_title("Stego Image Quality (SSIM) vs \u03b3")
    axes[1].legend(title="message size", fontsize=7, title_fontsize=8)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / "gamma_vs_stego_quality.png", bbox_inches="tight")
    plt.show()
else:
    print("No param_sweep data.")

## 2.2  Trade-off: Gamma vs Quality vs Robustness (dual-axis)

In [ ]:
if len(ps) > 0:
    # Fix message length to 512 for a clean plot
    sub512 = ps[ps["message_length"] == 512]
    if len(sub512) == 0:
        sub512 = ps  # fallback

    agg = sub512.groupby("gamma").agg(
        psnr_mean=("psnr_stego", "mean"),
        ssim_mean=("ssim_stego", "mean"),
        acc_mean=("bit_accuracy_pct", "mean"),
        acc_std=("bit_accuracy_pct", "std"),
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    # Quality on left axis
    l1, = ax1.plot(agg["gamma"], agg["psnr_mean"], "b-o", label="PSNR (stego quality)")
    ax1.set_xlabel("Gamma (\u03b3)")
    ax1.set_ylabel("PSNR (dB)", color="blue")
    ax1.tick_params(axis="y", labelcolor="blue")

    # Bit accuracy on right axis
    l2 = ax2.errorbar(agg["gamma"], agg["acc_mean"], yerr=agg["acc_std"],
                       color="green", marker="s", capsize=4, label="Bit Accuracy (%)")
    ax2.set_ylabel("Bit Accuracy (%)", color="green")
    ax2.tick_params(axis="y", labelcolor="green")
    ax2.axhline(y=50, color="red", ls="--", alpha=0.3)

    # Combined legend
    lines = [l1, l2]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="center right", fontsize=9)

    ax1.set_title(f"Trade-off: \u03b3 \u2014 Image Quality vs Robustness  ({bits_chars_label(512)}, JPEG q=75)")
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "gamma_tradeoff_quality_vs_robustness.png", bbox_inches="tight")
    plt.show()
else:
    print("No param_sweep data.")

In [ ]:
if len(ps) > 0:
    unique_lengths = sorted(ps["message_length"].unique())
    n = len(unique_lengths)

    if n == 0:
        print("No message lengths found.")
    else:
        import math

        # --- Grid size ---
        cols = 2
        rows = math.ceil(n / cols)

        fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))

        # Make axes iterable in all cases
        if rows == 1 and cols == 1:
            axes = [[axes]]
        elif rows == 1:
            axes = [axes]
        elif cols == 1:
            axes = [[ax] for ax in axes]

        axes_flat = [ax for row in axes for ax in row]

        for idx, msg_len in enumerate(unique_lengths):
            ax1 = axes_flat[idx]
            ax2 = ax1.twinx()

            sub = ps[ps["message_length"] == msg_len]

            agg = sub.groupby("gamma").agg(
                psnr_mean=("psnr_stego", "mean"),
                ssim_mean=("ssim_stego", "mean"),
                acc_mean=("bit_accuracy_pct", "mean"),
                acc_std=("bit_accuracy_pct", "std"),
            ).reset_index()

            # --- PSNR (blue) ---
            l1, = ax1.plot(
                agg["gamma"],
                agg["psnr_mean"],
                color="blue",
                marker="o",
                label="PSNR (dB)"
            )
            ax1.set_xlabel("Gamma (γ)")
            ax1.set_ylabel("PSNR (dB)", color="blue")
            ax1.tick_params(axis="y", labelcolor="blue")

            # --- Bit Accuracy (green) ---
            l2 = ax2.errorbar(
                agg["gamma"],
                agg["acc_mean"],
                yerr=agg["acc_std"],
                color="green",
                marker="s",
                capsize=4,
                label="Bit Accuracy (%)"
            )
            ax2.set_ylabel("Bit Accuracy (%)", color="green")
            ax2.tick_params(axis="y", labelcolor="green")
            ax2.axhline(y=50, color="red", linestyle="--", alpha=0.3)

            ax1.set_title(f"{bits_chars_label(msg_len)}")

            # Combined legend
            lines = [l1, l2]
            labels = [l.get_label() for l in lines]
            ax1.legend(lines, labels, loc="best", fontsize=8)

        # Hide unused subplots (if grid bigger than needed)
        for j in range(len(unique_lengths), len(axes_flat)):
            fig.delaxes(axes_flat[j])

        fig.suptitle(
            "Trade-off: γ — Image Quality vs Robustness (JPEG q=75)",
            fontsize=14
        )

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(SAVE_DIR / "gamma_tradeoff_all_lengths_grid.png",
                    bbox_inches="tight")
        plt.show()

else:
    print("No param_sweep data.")


## 2.3  Scatter: Stego PSNR vs Bit Accuracy (colored by gamma)

In [ ]:
if len(ps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc0 = axes[0].scatter(
        ps["psnr_stego"], ps["bit_accuracy_pct"],
        c=ps["gamma"], cmap="viridis", alpha=0.7, s=30, edgecolors="k", linewidths=0.3,
    )
    fig.colorbar(sc0, ax=axes[0], label="\u03b3")
    axes[0].set_xlabel("PSNR stego (dB)")
    axes[0].set_ylabel("Bit Accuracy (%)")
    axes[0].set_title("PSNR (stego) vs Bit Accuracy")
    axes[0].axhline(y=50, color="red", ls="--", alpha=0.3)

    sc1 = axes[1].scatter(
        ps["ssim_stego"], ps["bit_accuracy_pct"],
        c=ps["gamma"], cmap="viridis", alpha=0.7, s=30, edgecolors="k", linewidths=0.3,
    )
    fig.colorbar(sc1, ax=axes[1], label="\u03b3")
    axes[1].set_xlabel("SSIM stego")
    axes[1].set_ylabel("Bit Accuracy (%)")
    axes[1].set_title("SSIM (stego) vs Bit Accuracy")
    axes[1].axhline(y=50, color="red", ls="--", alpha=0.3)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / "stego_quality_vs_bit_acc_by_gamma.png", bbox_inches="tight")
    plt.show()
else:
    print("No param_sweep data.")

## 2.4  Correlation Matrix — All Numeric Variables

In [ ]:
num_cols = ["gamma", "message_length", "bit_accuracy", "ber",
            "psnr_stego", "ssim_stego", "psnr_transformed", "ssim_transformed"]
present = [c for c in num_cols if c in df.columns]
corr = df[present].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix — All Benchmark Variables")
plt.tight_layout()
plt.savefig(SAVE_DIR / "correlation_matrix.png", bbox_inches="tight")
plt.show()

# Print key correlations
print("\nKey correlations with bit_accuracy:")
if "bit_accuracy" in corr:
    ba_corr = corr["bit_accuracy"].drop("bit_accuracy").sort_values(key=abs, ascending=False)
    for var, val in ba_corr.items():
        print(f"  {var:25s}  r = {val:+.3f}")

## 2.5  Gamma vs Stego Quality — Marginal Distributions (joint plot)

In [ ]:
if len(ps) > 0:
    g = sns.jointplot(
        data=ps, x="psnr_stego", y="bit_accuracy_pct",
        hue="gamma", palette="viridis", alpha=0.6, s=30,
        height=7, marginal_kws={"fill": True, "alpha": 0.4},
    )
    g.ax_joint.axhline(y=50, color="red", ls="--", alpha=0.3)
    g.ax_joint.set_xlabel("PSNR stego (dB)")
    g.ax_joint.set_ylabel("Bit Accuracy (%)")
    g.figure.suptitle("Joint Distribution: Stego Quality vs Bit Accuracy", y=1.02)
    plt.savefig(SAVE_DIR / "joint_psnr_bit_acc.png", bbox_inches="tight")
    plt.show()
else:
    print("No param_sweep data.")

## 2.6  Statistical Summary — Gamma Effect

In [ ]:
if len(ps) > 0:
    gamma_summary = (
        ps.groupby("gamma")
        .agg(
            n=("ber", "count"),
            bit_acc_mean=("bit_accuracy_pct", "mean"),
            bit_acc_std=("bit_accuracy_pct", "std"),
            psnr_stego_mean=("psnr_stego", "mean"),
            psnr_stego_std=("psnr_stego", "std"),
            ssim_stego_mean=("ssim_stego", "mean"),
            ssim_stego_std=("ssim_stego", "std"),
        )
        .round(3)
    )

    print("Gamma Effect Summary (param_sweep suite, JPEG q=75)")
    print("=" * 80)
    display(gamma_summary)

    # Pearson correlation: gamma vs psnr_stego
    r_psnr, p_psnr = sp_stats.pearsonr(ps["gamma"], ps["psnr_stego"])
    r_ssim, p_ssim = sp_stats.pearsonr(ps["gamma"], ps["ssim_stego"])
    r_acc,  p_acc  = sp_stats.pearsonr(ps["gamma"], ps["bit_accuracy"])

    print(f"\nPearson correlations:")
    print(f"  gamma vs PSNR_stego:    r = {r_psnr:+.4f}  (p = {p_psnr:.2e})")
    print(f"  gamma vs SSIM_stego:    r = {r_ssim:+.4f}  (p = {p_ssim:.2e})")
    print(f"  gamma vs bit_accuracy:  r = {r_acc:+.4f}   (p = {p_acc:.2e})")

    if r_psnr < 0:
        print("\n=> Higher gamma DECREASES stego image quality (lower PSNR).")
    else:
        print("\n=> Higher gamma does NOT decrease stego image quality (unexpected).")

    if r_acc > 0:
        print("=> Higher gamma INCREASES bit accuracy (more robust).")
    else:
        print("=> Higher gamma does NOT increase bit accuracy.")

    print("=> This confirms a quality-robustness trade-off governed by gamma.")
else:
    print("No param_sweep data.")

---
## Summary of Findings

In [ ]:
print("ANALYSIS SUMMARY")
print("=" * 60)

# 1. Best/worst bit accuracy across all attacks
agg_all = df.groupby(["suite", "name"])["bit_accuracy_pct"].mean().sort_values(ascending=False)
print("\n[Bit Accuracy Ranking — Top 5]")
for (suite, name), val in agg_all.head(5).items():
    print(f"  {suite:14s} | {name:20s} | {val:.1f}%")

print("\n[Bit Accuracy Ranking — Bottom 5]")
for (suite, name), val in agg_all.tail(5).items():
    print(f"  {suite:14s} | {name:20s} | {val:.1f}%")

# 2. Gamma trade-off
if len(ps) > 0:
    g_acc = ps.groupby("gamma")["bit_accuracy_pct"].mean()
    g_psnr = ps.groupby("gamma")["psnr_stego"].mean()
    best_g = g_acc.idxmax()
    print(f"\n[Gamma Trade-off]")
    print(f"  Best gamma for accuracy: {best_g}  ({g_acc[best_g]:.1f}% avg)")
    print(f"  ... but stego PSNR at that gamma: {g_psnr[best_g]:.1f} dB")
    sweet_spot = g_acc[g_acc > 80].index
    if len(sweet_spot) > 0:
        sweet_g = sweet_spot[0]  # smallest gamma above 80% acc
        print(f"  Smallest gamma with >80% accuracy: {sweet_g}  (PSNR={g_psnr[sweet_g]:.1f} dB)")

# 3. Capacity limit — show both bits and chars
if len(ps) > 0:
    print(f"\n[Max Capacity at >=90% Accuracy per Gamma]")
    for g in sorted(ps["gamma"].unique()):
        sub = ps[ps["gamma"] == g]
        good = sub[sub["bit_accuracy_pct"] >= 90].groupby("message_length").size()
        max_bits = good.index.max() if len(good) > 0 else 0
        max_chars = max_bits // 8
        print(f"  \u03b3={g}: {max_bits} bits ({max_chars} chars)")